In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd

In [ ]:
df = gpd.read_file('geolayers/Ленинградская область.geojson')
migr = gpd.read_file('geolayers/Миграция Ленинградская область.geojson')
pred = pd.read_csv('predictions.csv')

In [ ]:
# migr['Численность населения (чел.)'].hist()

In [ ]:
dis = pred.groupby('name_o')['d'].mean()
dis.hist()

In [ ]:
np.log(pred.sort_values('total_pop_flow')['total_pop_flow']).hist()

In [ ]:
df.geometry.explore()

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point


# Формируем список всех городов
cities = pd.DataFrame({"city": pd.concat([pred["name_o"], pred["name_d"]]).unique()})

# Добавляем координаты
coords_o = pred.groupby("name_o")[["lat_o", "lon_o"]].first().reset_index()
coords_d = pred.groupby("name_d")[["lat_d", "lon_d"]].first().reset_index()

cities = cities.merge(coords_o, left_on="city", right_on="name_o", how="left").drop(columns=["name_o"])
cities = cities.merge(coords_d, left_on="city", right_on="name_d", how="left").drop(columns=["name_d"])

# Объединяем координаты
cities["lat"] = cities["lat_o"].fillna(cities["lat_d"])
cities["lon"] = cities["lon_o"].fillna(cities["lon_d"])
cities = cities.drop(columns=["lat_o", "lon_o", "lat_d", "lon_d"])

# Считаем уехавших и приехавших
departures = pred.groupby("name_o")["total_pop_flow"].sum().reset_index().rename(columns={"name_o": "city", "total_pop_flow": "уехало"})
arrivals = pred.groupby("name_d")["total_pop_flow"].sum().reset_index().rename(columns={"name_d": "city", "total_pop_flow": "приехало"})

# Объединяем данные
cities = cities.merge(departures, on="city", how="left").merge(arrivals, on="city", how="left").fillna(0)

# Создаем геометрию точек
cities["geometry"] = cities.apply(lambda row: Point(row["lon"], row["lat"]), axis=1)

# Преобразуем в GeoDataFrame
gdf = gpd.GeoDataFrame(cities, geometry="geometry", crs="EPSG:4326")

In [ ]:
# Присоединяем города к полигонам (пространственный join)
gdf = gdf.sjoin(df, how="left", predicate="within")

# Группируем по полигону и считаем сумму уехало/приехало
polygon_stats = gdf.groupby("index_right")[["уехало", "приехало"]].sum().reset_index()

# Объединяем обратно с полигонами
result = df.merge(polygon_stats, left_index=True, right_on="index_right", how="left").fillna(0)


In [ ]:
result.sort_values('уехало', ascending=False)

In [ ]:
result['dif'] = (result['приехало'] - result['уехало']) /1000

In [ ]:
result.explore(column='dif')

In [ ]:
migr['dif'] = migr['Количество приехавших'] - migr['Количество уехавших']
migr[['Количество уехавших','Количество приехавших', 'dif', 'geometry']][migr['name'].isna()][1:].explore(column='dif')